# J1-01 · Infrastructure locale & tracing dès la première ligne

**Formation « Orchestration d'agents & Monitoring : LangGraph + Langfuse (stack 100 % locale) »**
Jour 1 · créneau 09h15–10h30 (Lab 0) · ⏱️ ~1h15 (dont 45 min de pratique)

| | |
|---|---|
| 🎯 **Objectifs** | 1. Vérifier que la stack locale (Ollama + Langfuse v4) répond · 2. Tracer un premier appel LLM · 3. Regrouper des appels en **session** · 4. Instrumenter son **propre code Python** · 5. **Masquer les données personnelles** avant qu'elles n'atteignent les traces |
| 🧱 **Fil rouge** | Première brique de *HelpDesk-Ops* : le **classifieur de tickets** de NordTech SA (entreprise fictive) |
| ✅ **Prérequis** | `uv sync` et `uv run generate_env.py` faits, `docker compose up -d` lancé, `uv run smoke_test.py` au vert, notebook ouvert via `uv run --project infrastructure jupyter lab` (depuis la racine du dossier `Orchestration`) |
| 📦 **Versions** | `langchain 1.4` · `langgraph 1.2` · `langfuse 4.15` (SDK **v4**) · Ollama 0.34 · Qdrant 1.19 · uv 0.12 : figées par `uv.lock` le 28/09/2026, revérifiées le 20/10/2026 (Langfuse SDK v5 et Ollama 0.40 sont sortis depuis, volontairement non utilisés : voir section 4) |

## 1. Pourquoi tracer dès le jour 1 ?

### [Théorie] L'agent est une boîte noire, alors installons l'enregistreur de vol

Un avion de ligne embarque une **boîte noire** depuis son tout premier vol, et pas seulement « quand il y aura un problème ». Le jour où quelque chose tourne mal, on ne peut plus revenir en arrière pour enregistrer.

Un agent LLM, c'est pareil, en pire :

| Logiciel classique | Application LLM / agent |
|---|---|
| Même entrée → même sortie | **Non déterministe** : même question, réponses différentes |
| Un bug = une exception | Un « bug » = une réponse **plausible mais fausse**, sans exception |
| Les logs suffisent | Il faut voir **le prompt réel envoyé**, les **outils appelés**, les **tokens** et la **latence de chaque étape** |
| Coût marginal ≈ 0 | Chaque appel coûte du temps GPU (et de l'argent en cloud) |

C'est pour ça que **Langfuse est branché dans ce premier notebook**, et non au jour 4. Toute la semaine, chaque exercice laissera une trace qu'on pourra analyser, comparer et noter.

> **Le marché le confirme :** d'après le rapport *State of Agent Engineering* (LangChain, publié en juin 2026, enquête de nov.–déc. 2025), **89 %** des organisations qui ont des agents ont mis en place de l'observabilité, **94 %** parmi celles qui sont en production.

> 💡 **Le « Tips » du Formateur :** une erreur classique consiste à déboguer un agent avec des `print()`. Ça marche pour 1 appel, pas pour un graphe de 12 nœuds qui boucle 3 fois. Prenez l'habitude dès maintenant : **« je ne comprends pas ce que fait mon agent » → j'ouvre la trace, pas mon code**.

### Architecture de la stack (tout tourne sur votre poste)

```mermaid
flowchart LR
    NB["📓 Notebook Python<br/>LangChain / LangGraph"] -- "HTTP :11434<br/>prompts" --> OL["🦙 Ollama<br/>granite4.1 · gemma4"]
    NB -- "HTTP :6333<br/>vecteurs (clé API)" --> QD["🧭 Qdrant<br/>base vectorielle"]
    NB -- "OTLP/HTTP :3000<br/>traces (SDK v4)" --> LFW["🪢 Langfuse web"]
    LFW --> R[(Redis)] --> LFK["Langfuse worker"]
    LFK --> CH[(ClickHouse<br/>traces)]
    LFW --> PG[(Postgres<br/>projets, prompts)]
    LFW --> S3[(MinIO<br/>événements bruts)]
```

**Aucune flèche ne sort de la machine.** C'est l'argument « souveraineté » que vous pourrez défendre face à un DPO ou à un RSSI : pas de clé d'API, pas de SaaS, pas de transfert hors UE.

## 2. Charger la configuration

### [Code] Lire `.env` (une seule source de vérité)

Les noms de modèles et les clés Langfuse viennent du fichier `infrastructure/.env` généré par `generate_env.py`. **Aucun secret n'est écrit dans le notebook.**

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv


def trouver_env(nom_dossier: str = "infrastructure") -> Path:
    """Remonte l'arborescence depuis le notebook jusqu'à trouver infrastructure/.env."""
    for dossier in [Path.cwd(), *Path.cwd().parents]:
        candidat = dossier / nom_dossier / ".env"
        if candidat.exists():
            return candidat
    raise FileNotFoundError("infrastructure/.env introuvable : lance `uv run generate_env.py`")


ENV_PATH = trouver_env()
load_dotenv(ENV_PATH, override=True)  # override=True : si tu modifies .env, relancer la cellule suffit

OLLAMA_BASE_URL = os.environ["OLLAMA_BASE_URL"]
MODEL_AGENT = os.environ["MODEL_AGENT"]

# ⚠️ SDK Langfuse v4 : la variable s'appelle LANGFUSE_BASE_URL (LANGFUSE_HOST est déprécié)
print(f".env chargé depuis  : {ENV_PATH}")
print(f"Ollama              : {OLLAMA_BASE_URL}")
print(f"Modèle agent        : {MODEL_AGENT}")
print(f"Langfuse            : {os.environ['LANGFUSE_BASE_URL']}")
# On n'affiche QUE le début de la clé publique : un secret ne s'imprime jamais, même en formation
print(f"Clé publique        : {os.environ['LANGFUSE_PUBLIC_KEY'][:10]}…")

**👀 Ce que vous devez observer :** 5 lignes, sans erreur. `KeyError` → la variable manque dans `.env` ; `FileNotFoundError` → `generate_env.py` n'a pas été lancé.

## 3. Ollama : le moteur local

### [Théorie] Ollama = un « Docker pour modèles »

Ollama télécharge, stocke et sert des modèles via une API HTTP locale (`:11434`), comme Docker le fait pour les images. Deux réglages comptent pour les agents :

- **`temperature=0`** : on veut des décisions **reproductibles** (choisir un outil, classer un ticket), pas de la créativité.
- **`num_ctx`** (taille de contexte) : un agent accumule messages + résultats d'outils. Un contexte trop petit fait **tronquer le début de la conversation en silence**, et donc disparaître les instructions système.

> 💡 **Le « Tips » du Formateur :** le premier appel à un modèle est **lent** (chargement en mémoire : 5 à 30 s selon la machine). Les suivants sont rapides tant que le modèle reste chargé (`OLLAMA_KEEP_ALIVE=30m`). Ne concluez jamais « le modèle est lent » sur le premier appel. On le verra noir sur blanc dans Langfuse.

In [ ]:
import httpx

# Interroge directement l'API Ollama : quels modèles sont disponibles localement ?
reponse = httpx.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
reponse.raise_for_status()

for modele in reponse.json()["models"]:
    taille_go = modele["size"] / 1e9
    print(f"  • {modele['name']:<22} {taille_go:5.1f} Go")

In [ ]:
import time

from langchain_ollama import ChatOllama

llm = ChatOllama(
    model=MODEL_AGENT,
    base_url=OLLAMA_BASE_URL,
    temperature=0,      # décisions reproductibles
    num_ctx=16384,      # contexte suffisant pour les agents des jours suivants
)

# On mesure 2 appels identiques : le 1er inclut le chargement du modèle en mémoire
for essai in (1, 2):
    debut = time.perf_counter()
    message = llm.invoke("En une phrase : qu'est-ce qu'un ticket de support informatique ?")
    duree = time.perf_counter() - debut

    # response_metadata contient les compteurs bruts renvoyés par Ollama
    meta = message.response_metadata
    print(f"Essai {essai} : {duree:5.1f}s | tokens générés : {meta.get('eval_count')}")

print("\nRéponse :", message.content)

**👀 Ce que vous devez observer :** l'essai 1 est nettement plus lent que l'essai 2 (chargement du modèle). Notez les deux durées : on les retrouvera dans Langfuse dans un instant.

## 4. Langfuse v4 : le vocabulaire en 2 minutes

### [Théorie] Trace, observation, session

*Analogie : un ticket de caisse détaillé.* La **trace** est le ticket entier (une requête utilisateur de bout en bout). Chaque **observation** est une ligne du ticket (un appel LLM, un outil, une étape de code). La **session** est la carte de fidélité : elle relie tous les tickets d'un même client.

| Concept | Ce que c'est | Exemple HelpDesk-Ops |
|---|---|---|
| **Trace** | une exécution de bout en bout | « classer le ticket #1042 » |
| **Observation** | une étape, **typée** en v4 : `span`, `generation`, `tool`, `agent`, `chain`, `retriever`, `guardrail`… | l'appel au LLM, la recherche dans la KB |
| **Session** | plusieurs traces d'une même conversation | les 3 échanges avec Mme Dupont |
| **User** | l'utilisateur final | `agent-n1-042` |
| **Tags / metadata** | étiquettes pour filtrer | `jour-1`, `classifieur` |
| **Score** | une note (humaine, automatique, LLM-juge) | 👍 / 👎, justesse = 0,8 → **jour 4** |

Sous le capot, le SDK v4 est construit sur **OpenTelemetry**, le standard ouvert de l'observabilité. Ce que vous apprenez ici se transpose à Grafana, Datadog, etc.

> ⚠️ **Le « Tips » du Formateur : attention aux tutoriels obsolètes.** L'API Langfuse a changé **trois fois** (v2 → v3 → v4 → v5). La majorité des exemples en ligne, et des réponses de ChatGPT et consorts, ne correspondent pas à la version installée :
>
> | ❌ À bannir | ✅ SDK v4 (cette formation) |
> |---|---|
> | `from langfuse.callback import CallbackHandler` (v2) | `from langfuse.langchain import CallbackHandler` |
> | `CallbackHandler(public_key=…, secret_key=…)` (v2) | clés lues depuis l'environnement, `CallbackHandler()` |
> | `CallbackHandler(update_trace=True)` (v3) | `with propagate_attributes(...)` |
> | `langfuse.update_current_trace(session_id=…)` (v3) | `with propagate_attributes(session_id=…)` |
> | `start_as_current_span` / `start_as_current_generation` (v3) | `start_as_current_observation(as_type=…)` |
> | `LANGFUSE_HOST` | `LANGFUSE_BASE_URL` |
>
> **Et la v5 ?** Sortie en octobre 2026, elle exige un serveur Langfuse ≥ 4.59 et change notamment `run_experiment` (via OpenTelemetry) et l'encodage des metadata. Le `pyproject.toml` borne `langfuse<5` : le code de cette formation reste en v4. Vérifiez toujours la version (`uv pip show langfuse`) avant de copier un exemple.

### [Code] Créer le client Langfuse, avec protection des données dès la conception

On crée le client **une seule fois**, au démarrage. On y branche tout de suite une fonction de **masquage des données personnelles** : c'est le principe de *privacy by design* (RGPD, art. 25). On détaille ce masquage en section 7.

> 💡 **Le « Tips » du Formateur :** le SDK v4 garde **un client unique par clé publique** (singleton). La première configuration gagne. Si vous modifiez `masquer_pii` plus tard, **redémarrez le kernel**.
>
> 🐛 **Retour d'expérience (vécu en préparant ce notebook) :** la première version du masque gérait `str`, `dict` et `list`, mais pas les `tuple`. Or `@observe` enregistre les arguments de la fonction sous la forme `{"args": (…,), "kwargs": {…}}`, donc un **tuple**. Résultat : l'e-mail et l'IBAN partaient **en clair** dans la trace racine, alors que les sous-étapes étaient bien masquées. D'où les 3 `assert` de la cellule : **un masque se teste comme n'importe quel code de sécurité**.

In [ ]:
import re

from langfuse import Langfuse

# --- Masquage des données personnelles AVANT envoi vers Langfuse --------------------
# Expressions volontairement simples (pédagogiques). En production : une bibliothèque
# dédiée (ex. Microsoft Presidio) ou le PIIMiddleware de LangChain (jour 5).
MOTIFS_PII = {
    "EMAIL": re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+"),
    "TEL": re.compile(r"(?:\+32|0)\s?4\d{2}(?:[\s./]?\d{2}){3}"),   # mobiles belges : 04xx xx xx xx
    "IBAN": re.compile(r"\bBE\d{2}(?:\s?\d{4}){3}\b"),              # IBAN belge
}


def masquer_pii(*, data, **kwargs):
    """Appelée par Langfuse sur chaque input/output/metadata avant l'envoi.

    Reçoit n'importe quel type (str, dict, list…) : on descend récursivement
    et on ne remplace que dans les chaînes de caractères.
    """
    if isinstance(data, str):
        for etiquette, motif in MOTIFS_PII.items():
            data = motif.sub(f"<{etiquette}>", data)
        return data
    if isinstance(data, dict):
        return {cle: masquer_pii(data=valeur) for cle, valeur in data.items()}
    if isinstance(data, (list, tuple)):
        # ⚠️ tuple indispensable : @observe transmet les arguments de fonction sous forme de tuple
        return type(data)(masquer_pii(data=element) for element in data)
    return data


# Tests unitaires minimaux : un masque non testé est un masque qui fuit
assert masquer_pii(data="écrire à a.b@c.be") == "écrire à <EMAIL>"
assert masquer_pii(data={"k": ["0470 12 34 56"]}) == {"k": ["<TEL>"]}
assert masquer_pii(data=("BE71 0961 2345 6769",)) == ("<IBAN>",)

# Clés et URL lues automatiquement depuis l'environnement (LANGFUSE_PUBLIC_KEY, _SECRET_KEY, _BASE_URL)
langfuse = Langfuse(mask=masquer_pii)

assert langfuse.auth_check(), "Langfuse refuse les clés (ou démarre encore : attendre 2-3 min)"
print("✅ Connecté à Langfuse, masquage PII actif")

## 5. Première brique du fil rouge : le classifieur de tickets tracé

### [Théorie] Le `CallbackHandler`, un espion branché sur LCEL

LangChain émet des **événements** à chaque étape d'une chaîne (début, fin, tokens, erreur). Le `CallbackHandler` de Langfuse les écoute et les transforme en observations. On le passe dans le `config` de l'appel, **sans modifier la chaîne**. C'est le pattern *Observer*, appliqué à l'IA.

> 💡 **Le « Tips » du Formateur :** passez le handler **à l'appel** (`config={"callbacks": [...]}`) plutôt qu'au constructeur du modèle. Le handler suit alors toute la chaîne (prompt → LLM → parser), sous-étapes comprises.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langfuse.langchain import CallbackHandler

CATEGORIES = ["reseau", "materiel", "logiciel", "acces", "autre"]

# Prompt système : rôle + règles + format de sortie strict (un seul mot)
prompt_classifieur = ChatPromptTemplate.from_messages([
    ("system",
     "Tu es l'agent de tri N1 du help-desk de NordTech SA. "
     "Classe le ticket dans UNE catégorie parmi : {categories}. "
     "Réponds uniquement par le nom de la catégorie, en minuscules, sans ponctuation."),
    ("human", "Ticket : {ticket}"),
])

# LCEL : prompt | modèle | parser
classifieur = prompt_classifieur | llm | StrOutputParser()

langfuse_handler = CallbackHandler()   # v4 : aucun argument nécessaire

categorie = classifieur.invoke(
    {"ticket": "Impossible d'accéder au VPN depuis ce matin", "categories": ", ".join(CATEGORIES)},
    config={
        "callbacks": [langfuse_handler],
        "run_name": "classifieur-ticket",   # nom lisible de la trace dans l'UI
    },
)
print("Catégorie prédite :", categorie)

**👀 Ce que vous devez observer :** ouvrez **http://localhost:3000 → Tracing**. Une trace `classifieur-ticket` apparaît, avec 3 sous-étapes : `ChatPromptTemplate` → `ChatOllama` (une **generation** : prompt complet, réponse, tokens, latence) → `StrOutputParser`.

Cliquez sur la generation : vous voyez **le prompt réellement envoyé**, variables remplacées. C'est l'information la plus utile pour déboguer un LLM.

*Rien n'apparaît ?* Le SDK envoie par **lots** en arrière-plan. Attendez quelques secondes ou exécutez `langfuse.flush()`.

## 6. Sessions, utilisateurs et tags avec `propagate_attributes`

### [Théorie] Étiqueter une fois, propager partout

En v4, les attributs de trace (session, user, tags, metadata, version) se posent avec un **gestionnaire de contexte** `with propagate_attributes(...)`. Tout ce qui s'exécute dans le bloc en hérite : appels LangChain, sous-étapes, code décoré.

*Analogie : le badge visiteur.* On le reçoit à l'accueil, et chaque porte franchie ensuite l'enregistre automatiquement.

> 💡 **Le « Tips » du Formateur :** en v4, la **metadata** est un `dict[str, str]` (valeurs ≤ 200 caractères). Passer un entier ou un objet est une erreur fréquente : convertissez en `str`. Et **jamais de donnée personnelle dans un tag** : les tags ne passent pas par tous les filtres et servent à filtrer des tableaux de bord partagés.

In [ ]:
from langfuse import propagate_attributes

tickets_session = [
    "Mon écran reste noir au démarrage",
    "Je n'ai plus les droits sur le dossier partagé Compta",
    "Outlook plante à l'ouverture d'une pièce jointe",
]

with propagate_attributes(
    session_id="demo-session-j1",          # regroupe les 3 traces
    user_id="agent-n1-042",                # pseudonyme de l'opérateur, jamais un nom réel
    tags=["jour-1", "classifieur"],
    metadata={"module": "J1-01", "modele": MODEL_AGENT},   # dict[str, str] !
    version="classifieur-v1",              # permettra de comparer les versions au jour 4
):
    for ticket in tickets_session:
        categorie = classifieur.invoke(
            {"ticket": ticket, "categories": ", ".join(CATEGORIES)},
            config={"callbacks": [langfuse_handler], "run_name": "classifieur-ticket"},
        )
        print(f"{categorie:<10} ← {ticket}")

**👀 Ce que vous devez observer :** dans **Sessions**, une session `demo-session-j1` regroupe 3 traces. Dans **Tracing**, filtrez par tag `classifieur` ou par user `agent-n1-042`.

## 7. Instrumenter votre propre code avec `@observe`

### [Théorie] Tout n'est pas LangChain

Un vrai traitement de ticket, ce n'est pas qu'un appel LLM : il y a du **code métier** (normaliser, appliquer une règle, interroger une base). `@observe` transforme n'importe quelle fonction Python en observation. Les appels imbriqués forment un **arbre**. Pour une étape précise, `start_as_current_observation(as_type=…)` donne un contrôle plus fin.

> 💡 **Le « Tips » du Formateur :** typez vos observations (`as_type="tool"`, `"guardrail"`, `"retriever"`…). Dans l'UI, les filtres et les icônes suivent ce type. Au jour 3, sur un système à 4 agents, la différence entre une trace lisible et un mur d'étapes se joue là.

In [ ]:
from langfuse import observe


import unicodedata


def sans_accents(texte: str) -> str:
    """'Accès Réseau' -> 'Acces Reseau' : décompose les lettres accentuées puis retire les accents."""
    return "".join(c for c in unicodedata.normalize("NFKD", texte) if not unicodedata.combining(c))


def normaliser_categorie(brute: str) -> str:
    """Garde-fou : un LLM peut répondre 'Réseau.', 'Accès' ou 'catégorie : reseau'. On normalise."""
    nettoyee = sans_accents(brute).strip().lower().rstrip(".")
    for categorie in CATEGORIES:
        if categorie in nettoyee:
            return categorie
    return "autre"   # repli sûr plutôt qu'une valeur inventée


# Un garde-fou se teste : les accents sont le piège n°1 en français
assert normaliser_categorie("Accès.") == "acces"
assert normaliser_categorie("Réseau") == "reseau"
assert normaliser_categorie("je ne sais pas") == "autre"


@observe(name="traiter-ticket")   # observation racine = la trace
def traiter_ticket(ticket: str) -> dict:
    # Étape 1 : appel LLM (tracé via le CallbackHandler, imbriqué sous traiter-ticket)
    brute = classifieur.invoke(
        {"ticket": ticket, "categories": ", ".join(CATEGORIES)},
        config={"callbacks": [langfuse_handler]},
    )

    # Étape 2 : code métier, tracé comme un garde-fou
    with langfuse.start_as_current_observation(
        name="normalisation", as_type="guardrail", input=brute
    ) as etape:
        categorie = normaliser_categorie(brute)
        etape.update(output=categorie, metadata={"corrige": str(brute.strip() != categorie)})

    # Étape 3 : règle métier simple (demain, ce sera une arête conditionnelle LangGraph)
    file_attente = "N2-infra" if categorie in ("reseau", "materiel") else "N1"
    return {"categorie": categorie, "file": file_attente}


with propagate_attributes(session_id="demo-session-j1", tags=["jour-1", "observe"]):
    resultat = traiter_ticket("Le Wi-Fi de la salle de réunion B coupe toutes les 5 minutes")
print(resultat)

**👀 Ce que vous devez observer :** une trace `traiter-ticket` avec un **arbre** : la chaîne LangChain en dessous, puis l'observation `normalisation` de type *guardrail*. Comparez les latences : le code Python prend quelques millisecondes, le LLM plusieurs centaines. C'est quasiment toujours le LLM qui domine le temps de réponse.

## 8. 🔒 RGPD : ce qui entre dans une trace est une donnée personnelle

### [Théorie] Vos traces sont une base de données de conversations

Un ticket de support contient des **noms, e-mails, numéros de téléphone**, parfois pire. Tout ce qui est tracé est **stocké** (ClickHouse + MinIO) et **consultable** par toute personne ayant accès à l'UI. Au sens du RGPD, c'est un **traitement de données personnelles**, avec ses obligations :

| Obligation RGPD | Traduction technique |
|---|---|
| Minimisation (art. 5.1.c) | masquer avant envoi (`mask=`), ne tracer que ce qui sert |
| Protection dès la conception (art. 25) | masquage configuré **à la création du client**, pas après coup |
| Limitation de conservation (art. 5.1.e) | durée de conservation définie **et appliquée**. ⚠️ La *data retention* automatique de Langfuse est réservée à l'édition Enterprise (et aux offres Cloud Pro/Enterprise) : en auto-hébergé open source, il faut une **purge planifiée** et documentée |
| Sécurité (art. 32) | accès à l'UI restreint, pas d'exposition réseau, secrets hors du code |

**NIS2**, pour les entités concernées : ces mêmes traces sont une **preuve précieuse** pour analyser un incident et le notifier. Il faut donc les conserver *assez*, mais pas *trop*, et elles doivent être *propres*.

> 💡 **Le « Tips » du Formateur :** le masquage par regex est une **ceinture**, pas un parachute. Il rate les noms propres, les adresses postales, les fautes de frappe, et même des formats courants : nos motifs ignorent `+32 (0)470…`, `0032 470…`, les fixes (`02 123 45 67`) et les IBAN non belges. En entreprise, combinez regex + détection NER (Presidio) + **rétention courte** + **accès restreint**. Aucune de ces mesures ne suffit seule.

In [ ]:
ticket_avec_pii = (
    "Bonjour, c'est Julie Martin (julie.martin@nordtech.example, 0470 12 34 56). "
    "Mon remboursement sur BE71 0961 2345 6769 n'apparaît pas dans l'outil RH."
)

# Ce que VOUS voyez localement (le masquage ne s'applique qu'à l'envoi vers Langfuse)
print("Local    :", ticket_avec_pii)
print("Langfuse :", masquer_pii(data=ticket_avec_pii))

with propagate_attributes(session_id="demo-session-j1", tags=["jour-1", "rgpd"]):
    print("\nRésultat :", traiter_ticket(ticket_avec_pii))

**👀 Ce que vous devez observer :** dans l'UI, la trace montre `<EMAIL>`, `<TEL>`, `<IBAN>`, alors que le LLM local a bien reçu le texte complet.

⚠️ **Mais « Julie Martin » est toujours visible !** C'est exactement la limite des regex évoquée plus haut. Gardez cette observation : on corrigera au jour 5 avec le `PIIMiddleware`.

**Question pour la salle :** le masquage protège la *trace*. Protège-t-il aussi le *modèle* ? Le modèle est local : est-ce un problème ? Et s'il était dans le cloud ?

## 9. 🧪 Mini-défi (15 min)

Ajoutez une étape **`estimer_priorite`** au traitement :

1. Écrivez une fonction `estimer_priorite(ticket: str) -> str` qui renvoie `"P1"` si le ticket contient un mot d'urgence (`"bloqué"`, `"urgent"`, `"production"`, `"tous les utilisateurs"`), sinon `"P3"`. Pensez aux majuscules et aux accents (`sans_accents` est déjà défini).
2. Tracez-la comme une observation de type **`tool`** avec `start_as_current_observation`.
3. Intégrez la priorité au dictionnaire renvoyé par `traiter_ticket` (créez `traiter_ticket_v2`).
4. Exécutez les 3 tickets ci-dessous dans une **nouvelle session** `defi-j1-<vos initiales>`, avec la version `classifieur-v2` :
   - « URGENT : le serveur de production ERP est bloqué »
   - « Je voudrais un deuxième écran pour mon poste »
   - « Tous les utilisateurs du 3e étage ont perdu le réseau »
5. **Dans l'UI** : retrouvez votre session et identifiez l'**étape la plus lente** de chaque trace.

➡️ Corrigé : `exercices/corriges/J1_01_defi_corrige.ipynb`

In [ ]:
# À vous de jouer 👇

## 10. Avant de quitter le notebook : `flush()`

### [Théorie] Envoi asynchrone par lots

Le SDK n'envoie pas chaque observation immédiatement : il les **regroupe** et les expédie en arrière-plan, pour ne jamais ralentir votre application. Conséquence : dans un script court ou à la fermeture d'un notebook, **les derniers événements peuvent être perdus** si on ne force pas l'envoi.

> 💡 **Le « Tips » du Formateur :** dans une API (FastAPI, Streamlit), appelez `langfuse.shutdown()` à l'arrêt de l'application. Dans un job batch ou en CI, `langfuse.flush()` en fin de script. L'oubli de `flush()` est la **cause n°1** du « mes traces n'apparaissent pas ».

In [ ]:
langfuse.flush()   # force l'envoi des événements en attente
print("✅ Toutes les observations sont envoyées. Rendez-vous dans l'UI : http://localhost:3000")

## 🧭 Récap

| Vous savez maintenant… | API v4 |
|---|---|
| Configurer le client, avec masquage PII | `Langfuse(mask=…)`, `auth_check()` |
| Tracer une chaîne LangChain | `from langfuse.langchain import CallbackHandler` + `config={"callbacks": [...]}` |
| Regrouper en session, étiqueter | `with propagate_attributes(session_id=…, user_id=…, tags=…, metadata=…, version=…)` |
| Tracer votre propre code | `@observe`, `start_as_current_observation(as_type=…)` |
| Ne rien perdre | `flush()` / `shutdown()` |

**Et ensuite ?** Tout ce code (lecture de `.env`, masquage, client Langfuse, classifieur) est rangé dans le paquet `src/helpdesk_ops` (`config`, `pii`, `observabilite`, `tri`) : les notebooks suivants l'importent en 3 lignes au lieu de le recopier.

**Prochain notebook, J1-02 :** la boucle ReAct codée à la main, puis `create_agent`, en regardant dans Langfuse ce que l'agent fait vraiment.

### ✅ Quiz express

1. Pourquoi le premier appel à Ollama est-il plus lent que les suivants ?
2. En SDK v4, comment associe-t-on 3 appels à une même conversation ?
3. Vos traces n'apparaissent pas à la fin d'un script. Première chose à vérifier ?
4. Citez deux limites d'un masquage PII par expressions régulières.

<details>
<summary>Réponses</summary>

1. Le modèle est chargé en mémoire (RAM/VRAM) au premier appel ; il reste ensuite chargé pendant `OLLAMA_KEEP_ALIVE`.
2. `with propagate_attributes(session_id="…"):` autour des appels (et non plus `update_current_trace`, qui date de la v3).
3. L'appel à `langfuse.flush()` (envoi asynchrone par lots).
4. Il rate les noms propres et les formats imprévus (fautes, formats étrangers) ; il ne protège que la trace, pas le modèle ni les logs applicatifs.
</details>